# TRM Compiler Pass Ordering — Real LLVM Training

Train a 60K-param TRM model on LLVM optimization passes, benchmark vs LLVM -Oz/-O3.

**Two modes:**
- `--synthetic`: Runs immediately without CompilerGym (uses realistic synthetic environment)
- Real CompilerGym: Requires grpcio which is difficult to install on Colab

**Runtime:** Change runtime to GPU (T4) for training.

In [ ]:
# Cell 1: Clone repo
import os
import subprocess

PROJECT_DIR = '/content/trm-youtubevids'
VENV_DIR = '/content/trm-env'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Cree0618/trm-youtubevids.git', PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
# Cell 2: Create Python 3.11 venv (if needed) and run the script
# The script works in synthetic mode without needing CompilerGym

import subprocess
import os

PROJECT_DIR = '/content/trm-youtubevids'
VENV_DIR = '/content/trm-env'

# Create venv if it doesn't exist
if not os.path.exists(VENV_DIR):
    result = subprocess.run(['python3.11', '-m', 'venv', VENV_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Creating venv failed: {result.stderr}")
    else:
        print(f"Created venv at {VENV_DIR}")
else:
    print(f"Venv already exists at {VENV_DIR}")

VENV_PIP = f'{VENV_DIR}/bin/pip'
VENV_PYTHON = f'{VENV_DIR}/bin/python'

# Upgrade pip in venv
subprocess.run([VENV_PIP, 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)

# Install torch and numpy
subprocess.run([VENV_PIP, 'install', 'torch', 'numpy'], check=True)

# Verify installation
result = subprocess.run([VENV_PYTHON, '-c', 'import torch; import numpy; print(f"torch={torch.__version__}, numpy={numpy.__version__}")'], capture_output=True, text=True)
print(result.stdout.strip())

In [ ]:
# Cell 3: Run the TRM script in synthetic mode (no CompilerGym needed)
# This trains the model and benchmarks against LLVM -Oz, -O3, Random, and TRM

import subprocess

PROJECT_DIR = '/content/trm-youtubevids'
VENV_PYTHON = f'{PROJECT_DIR}/../trm-env/bin/python'

# Run the script with synthetic mode (no CompilerGym required)
cmd = [
    VENV_PYTHON,
    f'{PROJECT_DIR}/trm_compiler_real_llvm.py',
    '--synthetic',
    '--epochs', '15',
    '--episodes', '20',
    '--benchmarks', 'qsort', 'adpcm', 'blowfish', 'bzip2',
    '--max-steps', '20',
    '--batch-size', '64',
    '--num-random', '50',
    '--seed', '42'
]

result = subprocess.run(cmd, capture_output=False, text=True)
print(f"\nScript exited with code: {result}")

In [ ]:
# Cell 4: (Optional) If you want to try real CompilerGym, run this cell first
# This often fails due to grpcio compilation issues on Colab

# import subprocess
# VENV_PIP = '/content/trm-env/bin/pip'
# 
# # Try to install grpcio from source (may fail)
# result = subprocess.run([VENV_PIP, 'install', 'grpcio>=1.32.0,<1.44.0'], capture_output=True, text=True)
# if result.returncode != 0:
#     print("grpcio installation failed - use --synthetic mode instead")
# else:
#     # Install compiler_gym without deps
#     subprocess.run([VENV_PIP, 'install', 'compiler_gym', '--no-deps'], check=True)
#     print("CompilerGym installed successfully!")

In [ ]:
# Cell 5: Run with real CompilerGym (if successfully installed)
# import subprocess
# PROJECT_DIR = '/content/trm-youtubevids'
# VENV_PYTHON = f'{PROJECT_DIR}/../trm-env/bin/python'
# 
# cmd = [
#     VENV_PYTHON,
#     f'{PROJECT_DIR}/trm_compiler_real_llvm.py',
#     '--epochs', '15',
#     '--episodes', '20',
#     '--benchmarks', 'qsort', 'adpcm', 'blowfish', 'bzip2',
#     '--max-steps', '20',
#     '--batch-size', '64',
#     '--num-random', '50',
# ]
# subprocess.run(cmd, check=True)

## Results

The script outputs tables showing:
- **Reward**: log-ratio reward (higher is better)
- **Reduction**: instruction count reduction percentage

Algorithms compared:
- `LLVM-Oz`: Fixed LLVM -Oz pipeline
- `LLVM-O3`: Fixed LLVM -O3 pipeline  
- `Random(N)`: Random search with N trials
- `TRM-blind`: TRM without real compiler feedback
- `TRM-loop`: TRM with closed-loop compiler feedback